# 05 — Moshi LoRA pt-BR (Trilha B / spine) · Colab A100-80GB ou G4-96GB

Pipeline fiel ao `kyutai-labs/moshi-finetune` oficial (config/notebook lidos do
clone em `research/repos/moshi-finetune`, 2026-06-10). Pico ~39,6GB (r=128,
dur=100, batch16 em H100); **no Colab use A100-80 ou G4** — em A100-40 só com
batch=1 e duration menor (config do notebook oficial).

Dados: wav **estéreo** — canal ESQUERDO = voz do moshi/agente, DIREITO = usuário
— + um `.json` de transcrição com timestamps por arquivo (gerado pelo annotate).
Fontes pt-BR: (a) sintético do nosso pipeline (`tools/data/synth`, engine qwen3/
chatterbox-ptbr — rode em runtime separado, pins conflitam) e (b) diálogos G4
gravados. Smoke opcional: DailyTalkContiguous (EN, 14GB) valida o pipeline.

⚠️ Sessão Colab cai em 24h: `run_dir` fica no DRIVE (checkpoints persistem).

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
GH_TOKEN = userdata.get('GH_TOKEN')
import os; os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
!git clone https://{GH_TOKEN}@github.com/pedrocormann/TTS-ptbr.git /content/TTS-ptbr 2>/dev/null || (cd /content/TTS-ptbr && git pull)
!git clone https://github.com/kyutai-labs/moshi-finetune.git /content/moshi-finetune 2>/dev/null
# uv sync respeita o uv.lock (moshi@commit pinado + sphn certo) — 'pip install -e' solto
# quebra: moshi-finetune fixa torch==2.6 (downgrade do torch 2.9 do Colab) e sphn==0.1.12
# vs HEAD do moshi. Treino roda com 'uv run' (abaixo).
!pip -q install uv && cd /content/moshi-finetune && uv sync
%cd /content/TTS-ptbr

## 1. Dados estéreo pt-BR
Espera os wavs estéreo prontos no Drive (`TTS-ptbr-data/stereo_ptbr/`), vindos de:
`gen_dialogues.py → synth_tts.py (qwen3) → compose_stereo.py` (runtime separado)
e/ou diálogos gravados (G4). Smoke EN: descomente o bloco DailyTalk.

In [ ]:
import json, pathlib, sphn

DATA = pathlib.Path('/content/drive/MyDrive/TTS-ptbr-data/stereo_ptbr')
wavs = sorted(str(p) for p in DATA.glob('*.wav'))
assert wavs, f'sem wavs em {DATA} — rode o pipeline sintético antes'
durations = sphn.durations(wavs)
egs = pathlib.Path('/content/ds_ptbr.jsonl')
with egs.open('w') as f:
    for p, d in zip(wavs, durations):
        if d is not None:
            f.write(json.dumps({'path': p, 'duration': d}) + '\n')
print(len(wavs), 'wavs ·', sum(d or 0 for d in durations)/3600, 'h')

# --- SMOKE EN (valida pipeline antes do pt-BR; 14GB) ---
# from huggingface_hub import snapshot_download
# snapshot_download('kyutai/DailyTalkContiguous', repo_type='dataset',
#                   local_dir='/content/data/daily-talk-contiguous')
# egs = '/content/data/daily-talk-contiguous/dailytalk.jsonl'

## 2. Transcrições com timestamps (annotate oficial, pt)
Sintético do nosso pipeline JÁ TEM os .json (compose_stereo gera) — pule.
Áudio real/gravado: rode o annotate (whisper 'medium' — o código avisa que
large-v3 não é recomendado p/ estéreo).

In [ ]:
need = [json.loads(l)['path'] for l in open(egs)
        if not pathlib.Path(json.loads(l)['path']).with_suffix('.json').exists()]
print(len(need), 'wavs sem .json')
if need:
    !python /content/moshi-finetune/annotate.py {egs} --lang pt --whisper_model medium --local

## 3. Config de treino (base = example/moshi_7B.yaml oficial; run_dir no Drive)

In [ ]:
import torch, yaml
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
big = vram > 70   # A100-80 / G4-96
config = {
  'data': {'train_data': str(egs), 'eval_data': '', 'shuffle': True},
  'moshi_paths': {'hf_repo_id': 'kyutai/moshiko-pytorch-bf16'},
  'full_finetuning': False,
  'lora': {'enable': True, 'rank': 128, 'scaling': 2.0, 'ft_embed': False},
  'first_codebook_weight_multiplier': 100.0,
  'text_padding_weight': 0.5,
  'duration_sec': 100,
  'batch_size': 4 if big else 1,        # oficial Colab A100: 1; H100 batch16=39,6GB
  'max_steps': 2000,
  'gradient_checkpointing': True,
  'optim': {'lr': 2e-6, 'weight_decay': 0.1, 'pct_start': 0.05},
  'seed': 0, 'log_freq': 10, 'eval_freq': 100, 'do_eval': False,
  'do_ckpt': True, 'ckpt_freq': 100,    # checkpoint frequente: sessão pode cair
  'save_adapters': True,
  'overwrite_run_dir': True,   # train.py ABORTA se run_dir existe (não tem resume) — sobrescreve
  'run_dir': '/content/drive/MyDrive/TTS-ptbr-data/runs/moshi_lora_ptbr_v1',
}
yaml.safe_dump(config, open('/content/ptbr.yaml', 'w'))
print(f'GPU {vram:.0f}GB → batch_size={config["batch_size"]}')
# OOM? reduza batch_size; persistindo, reduza duration_sec (efeito colateral:
# modelo pode silenciar mais cedo na conversa — aviso do README oficial).

In [ ]:
import os
os.environ['CUDA_DEVICE_ORDER'] = 'PCI_BUS_ID'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
!cd /content/moshi-finetune && uv run torchrun --nproc-per-node 1 -m train /content/ptbr.yaml

## 4. Conversar com o modelo (gradio tunnel) — gate F4: pt-BR inteligível em FD?

In [ ]:
!pip -q install gradio
import pathlib
ckpts = sorted(pathlib.Path(config['run_dir']).glob('checkpoints/checkpoint_*'))
CK = ckpts[-1] / 'consolidated'
print('checkpoint:', CK)
!python -m moshi.server --gradio-tunnel \
    --lora-weight={CK}/lora.safetensors --config-path={CK}/config.json

## 5. Eval do gate F4 (REPLAN)
- Grave 2-3 min de conversa via gradio → WER round-trip + escuta (pt-BR emergiu?
  turn-taking segurou? PAD blowup?).
- Latência: `phase0/spike_c_moshi/smoke_moshi.py` no MESMO runtime (teto FD).
- Mimi freeze (decisivo #1, se ainda não rodou):
  `!python phase0/spike_c_moshi/mimi_ptbr_roundtrip.py --in-dir ptbr_clips --transcripts ptbr_clips/trans.jsonl`
- PASS ⇒ registrar em VIGIL-LOG + tech-stack; FAIL de fluência ⇒ CPT leve (SDumont).
- Próximo (F5): RL de interatividade — base `kyutai/moshika-rl-seamless` (CC-BY,
  saiu 2026-06-10) como init OU receita arXiv 2606.11167 sobre nosso LoRA.